<a href="https://colab.research.google.com/github/williamryanmckee-nwa/Deep-Learning/blob/main/10_10_Converting_TensorFlow_Model_to_TensorFlow_Lite_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Converting Tensorflow Model to TensorFlow Lite Model
- The TensorFlow Lite converter takes a TensorFlow model and generates
a TensorFlow Lite model (an optimized **FlatBuffer** format identified by the **tflite** file extension).


## Steps to be followed:
1. Import the required libraries
2. Create and save the model
3. Convert the Keras model to a TensorFlow lite model
4. Convert concrete functions

### Step 1: Import the required libraries
- Use the popular open-source machine learning framework for building and training neural networks, and import the **tensorflow** library.


In [6]:
import tensorflow as tf
import numpy as np

### Step 2: Create and save the model
- Create a sequential model using the Keras API in TensorFlow.
- Compile the model with the specified optimizer and loss function.
- Fit the model to the training data for a specified number of epochs.
- Save the trained model in the specified directory using the TensorFlow **SavedModel** format.


In [7]:
# Define the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(1,)),  # Use Input layer for the input shape
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=1)
])

# Compile the model
model.compile(optimizer='sgd', loss='mean_squared_error')

# Convert lists to NumPy arrays
x = np.array([-1, 0, 1])
y = np.array([-3, -1, 1])

# Train the model
model.fit(x=x, y=y, epochs=5)

# Save the model
tf.saved_model.save(model, 'saved_model_keras_dir')

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step - loss: 3.3209
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 3.1519
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 2.9776
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 2.8121
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 2.6549


__Observation:__

The code defines, trains, and saves a sequential Keras model in TensorFlow.

### Step 3: Convert the Keras model to a Tensorflow Lite model
- Save the TensorFlow Lite model to a file.



In [8]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
  f.write(tflite_model)

Saved artifact at '/tmp/tmp3gdwvt18'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor_3')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135877783183120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135877783191184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135877776740368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135877776743440: TensorSpec(shape=(), dtype=tf.resource, name=None)


__Observation:__

The updated code converts a Keras model into a TensorFlow Lite model and saves it to a file named **model.tflite**.






### Step 4: Convert concrete functions
- Create a model using low-level `tf.` APIs.
- Define the custom TensorFlow module named **Squared** that squares input values.
- Create an instance of the Squared module.
- Get the concrete function from the module's **__call__** method.
- Convert the concrete function to a TensorFlow Lite model.



In [9]:
class Squared(tf.Module):
  @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
  def __call__(self, x):
    return tf.square(x)

model = Squared()

concrete_func = model.__call__.get_concrete_function()

converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func],
                                                            model)
tflite_model = converter.convert()

In [10]:
result = model(tf.constant([2.0]))  # Example to square the value 2
print(result)  # Should print 4.0

tf.Tensor([4.], shape=(1,), dtype=float32)


**Observation**:

The provided code defines a TensorFlow module called **Squared** and converts it into a TensorFlow Lite model using the TensorFlow Lite converter.